# 8.22 Machine translation

Machine translation is meaning preservation under a new vocabulary, a new grammar, and often a new word order.

Translation sits where language modeling becomes bilingual. Conditional decoders, attention alignments, and BLEU-style overlap checks must balance fluency, adequacy, source coverage, and length.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def mt_ladder():
    return [
        {"name": "D1 lesson exact pair", "src": ["je", "mange"], "reference": ["I", "eat"], "candidate": ["I", "eat"], "attention": np.array([[0.9, 0.1], [0.1, 0.9]]), "difficulty": 1, "logps": [-1.2, -1.8], "lengths": [2, 4]},
        {"name": "D2 clean word-order pairs", "src": ["nous", "aimons", "riz"], "reference": ["we", "like", "rice"], "candidate": ["we", "like", "rice"], "attention": np.eye(3), "difficulty": 2, "logps": [-1.0, -1.4], "lengths": [3, 4]},
        {"name": "D3 reordering/agreement", "src": ["le", "chat", "noir"], "reference": ["the", "black", "cat"], "candidate": ["the", "cat", "dark"], "attention": np.array([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]]), "difficulty": 4, "logps": [-1.0, -1.7], "lengths": [2, 3]},
        {"name": "D4 alternates and morphology", "src": ["elle", "a", "fini", "vite"], "reference": ["she", "finished", "quickly"], "candidate": ["she", "finished", "has", "now"], "attention": np.array([[0.7, 0.1, 0.1, 0.1], [0.1, 0.2, 0.6, 0.1], [0.1, 0.1, 0.2, 0.6]]), "difficulty": 6, "logps": [-0.9, -1.9], "lengths": [2, 3]},
        {"name": "D5 dropped words and idioms", "src": ["il", "pleut", "des", "cordes", "ce", "soir"], "reference": ["it", "is", "raining", "heavily", "tonight"], "candidate": ["it", "rains", "soon"], "attention": np.array([[0.6, 0.2, 0.0, 0.0, 0.1, 0.1], [0.1, 0.6, 0.1, 0.1, 0.0, 0.1], [0.0, 0.0, 0.0, 0.0, 0.2, 0.8]]), "difficulty": 9, "logps": [-1.2, -2.0], "lengths": [3, 5]},
    ]


def bleu1(candidate, references, length_penalty=False):
    if isinstance(references[0], str):
        refs = [references]
    else:
        refs = references
    ref_counts = Counter()
    for ref in refs:
        ref_counts |= Counter(ref)
    cand_counts = Counter(candidate)
    matches = 0
    for token, count in cand_counts.items():
        matches += min(count, ref_counts.get(token, 0))
    precision = matches / len(candidate) if candidate else 0.0
    if not length_penalty:
        return precision
    best_ref_len = min(len(ref) for ref in refs)
    if len(candidate) >= best_ref_len:
        penalty = 1.0
    else:
        penalty = math.exp(1 - best_ref_len / max(len(candidate), 1))
    return precision * penalty


def coverage(attention):
    return attention.sum(axis=0) / max(attention.shape[0], 1)


## The concept, built once (D1)\n\nAn attention translator factors the target sequence as $$P(y_{1:n}\mid x_{1:m})=\prod_{t=1}^{n}P(y_t\mid y_{\lt t},c_t),\qquad c_t=\sum_i\alpha_{t,i}h_i,\qquad \operatorname{BLEU1}=\frac{\#\{\text{candidate unigrams in reference}\}}{\#\{\text{candidate unigrams}\}}$$\n\nThe lesson toy `je mange → I eat` can score BLEU1 = 1.0 only because the candidate and reference exactly match.

In [ ]:
lexical = np.array([[0.9, 0.1], [0.1, 0.9]])
assert lexical[0, 0] == 0.9
assert lexical[1, 1] == 0.9
assert int(np.argmax(lexical[0])) == 0
assert int(np.argmax(lexical[1])) == 1
reordered = [1, 0, 2]
assert reordered[0] != 0
attention = np.array([[0.8, 0.2], [0.1, 0.9], [0.4, 0.6]])
assert np.allclose(attention.sum(axis=1), 1.0)
assert list(np.argmax(attention, axis=1)) == [0, 1, 1]
short_score = -1.2 / (2 ** 0.6)
long_score = -1.8 / (4 ** 0.6)
assert round(short_score, 3) == -0.792
assert round(long_score, 3) == -0.783
assert long_score > short_score
assert bleu1(["I", "eat", "rice"], ["I", "eat", "rice"]) == 1.0
print("Lesson numbers verified")

Now build one reusable toy translator/evaluator. It returns a candidate, attention matrix, raw BLEU1, and a fixed score that uses references plus brevity/coverage checks.

In [ ]:
def translate_and_score(rung, references=None):
    candidate = rung["candidate"]
    if references is None:
        references = [rung["reference"]]
    raw_bleu = bleu1(candidate, references, length_penalty=False)
    fixed_bleu = bleu1(candidate, references, length_penalty=True)
    source_coverage = coverage(rung["attention"])
    adequacy = float((source_coverage > 0.05).mean())
    return candidate, rung["attention"], raw_bleu, fixed_bleu * adequacy

rung = mt_ladder()[0]
candidate, attn, raw_bleu, fixed_bleu = translate_and_score(rung)
assert candidate == ["I", "eat"]
assert raw_bleu == 1.0
assert fixed_bleu == 1.0
print(candidate, raw_bleu, fixed_bleu)

## The dataset ladder

The D1–D5 ladder is inline: D1 is the exact lesson pair, D2 is clean word-order preserving, D3 adds reordering/agreement, D4 adds alternate wording/morphology, and D5 uses dropped words and idioms.

In [ ]:
rungs = mt_ladder()
for rung in rungs:
    print(rung["name"], "src_len=", len(rung["src"]), "cand_len=", len(rung["candidate"]), "ref_len=", len(rung["reference"]))
    print("sample", " ".join(rung["src"]), "->", " ".join(rung["candidate"]))

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    candidate, attn, raw_bleu, fixed_bleu = translate_and_score(rung)
    rows.append((rung["name"], rung["difficulty"], raw_bleu, fixed_bleu, candidate))
for name, difficulty, raw_bleu, fixed_bleu, candidate in rows:
    print(f"{name:34s} difficulty={difficulty:2d} BLEU1={raw_bleu:.3f} fixed={fixed_bleu:.3f} cand={' '.join(candidate)}")
raw_curve = [row[2] for row in rows]
assert raw_curve[0] == 1.0
assert raw_curve[1] == 1.0
assert raw_curve[2] < raw_curve[1]
assert raw_curve[3] <= raw_curve[2]
assert raw_curve[4] < raw_curve[3]

## Results visualization

In [ ]:
fig, axes = plt.subplots(1, len(rungs), figsize=(16, 3))
raw_scores = []
fixed_scores = []
difficulties = []
for col, rung in enumerate(rungs):
    candidate, attn, raw_bleu, fixed_bleu = translate_and_score(rung)
    raw_scores.append(raw_bleu)
    fixed_scores.append(fixed_bleu)
    difficulties.append(rung["difficulty"])
    axes[col].imshow(attn, cmap="Greens", vmin=0, vmax=1)
    axes[col].set_title(rung["name"].split()[0])
    axes[col].set_xlabel("source")
    axes[col].set_ylabel("target")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(difficulties, raw_scores, marker="o", label="raw BLEU1")
plt.plot(difficulties, fixed_scores, marker="x", label="reference + length + coverage")
plt.xlabel("translation difficulty")
plt.ylabel("BLEU / adequacy score")
plt.title("BLEU falls as reordering and omissions grow")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## Pitfall on the hardest rung

The trap is computing BLEU on identical strings and celebrating 1.0. D5 uses an imperfect translation with dropped idiom words, then fixes evaluation with real references, a brevity penalty, and source coverage.

In [ ]:
hard = rungs[-1]
bad_identical_demo = bleu1(hard["candidate"], hard["candidate"])
raw_against_reference = bleu1(hard["candidate"], hard["reference"])
alternates = [hard["reference"], ["it", "is", "pouring", "tonight"]]
with_references = bleu1(hard["candidate"], alternates, length_penalty=True)
source_coverage = coverage(hard["attention"])
coverage_penalty = float((source_coverage > 0.05).mean())
fixed = with_references * coverage_penalty
print("wrong identical BLEU", bad_identical_demo)
print("reference BLEU", raw_against_reference)
print("length/reference/coverage fixed", fixed)
assert bad_identical_demo == 1.0
assert raw_against_reference < bad_identical_demo
assert fixed <= raw_against_reference

## Evaluate it + Practice

- Metric: BLEU1 with references, brevity, and coverage diagnostics.
- No-skill baseline: compare the candidate to itself, which always looks perfect.
- Cheap sanity check: every attention row sums to one.
- Ablation: remove length penalty and D5 short candidates look too good.
- Failure signals: BLEU stays flat at 1.0 on D3–D5, or uncovered source words remain invisible.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.